In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install transformers==4.52.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 71.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 47.0 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggin

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-Nemo-Instruct-2407"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00005-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/4.91G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [4]:
def generate_text(prompt, max_new_tokens=300, temperature=0.0):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    if temperature > 0:
        kwargs.update(do_sample=True, temperature=temperature, top_p=0.95)
    else:
        kwargs.update(do_sample=False)

    outputs = model.generate(**inputs, **kwargs)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [5]:
buggy_code = """
def calculate_average(numbers):
    total = sum(numbers)
    return total / len(numbers)

numbers = []
print(calculate_average(numbers))
"""

In [6]:
prompt = f"""
You are an expert Python debugging assistant.

Analyze the following Python code.

Python code:
{buggy_code}

Identify the main bug.

Return ONLY a valid JSON object.
Do not include markdown.
Do not include ```json.
Do not include any explanation outside the JSON.

The JSON must contain exactly these fields:

{{
    "bug_type": "One of: Runtime Error, Syntax Error, Logic Error, Type Error, Name Error, Index Error, Import Error",
    "severity": "One of: Low, Medium, High",
    "line": "The line number containing the bug",
    "error": "The Python error name, or None if there is no error",
    "root_cause": "The main cause of the bug",
    "explanation": "A simple explanation of why the bug happens",
    "suggested_fix": "How to fix the bug"
}}
"""

In [27]:
result = generate_text(prompt, max_new_tokens=300)

print(result)

{"bug_type": "Runtime Error", "severity": "High", "line": 4, "error": "ZeroDivisionError", "root_cause": "Division by zero", "explanation": "The function is trying to divide by the length of the 'numbers' list, which is empty.", "suggested_fix": "Check if the list is empty before performing the division."}


In [8]:
import json

def clean_json_response(result):
    start = result.find("{")
    end = result.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("No JSON object found in model response.")

    return result[start:end + 1]

print(clean_json_response(result))


{
    "bug_type": "Runtime Error",
    "severity": "High",
    "line": 5,
    "error": "ZeroDivisionError",
    "root_cause": "Trying to divide by zero",
    "explanation": "The function is trying to divide the total sum of an empty list by the length of the list, which is zero.",
    "suggested_fix": "Check if the list is empty before performing the division."
}


In [54]:
!pip install -q json-repair

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00


In [55]:
import json
from json_repair import repair_json

In [57]:
def strip_fences(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1] if "\n" in text else ""
        text = text.rsplit("```", 1)[0]
    return text

def prepare_test(source_code, test_code):
    test_code = strip_fences(test_code)
    src = ast.parse(source_code)
    src_names = {n.name for n in src.body
                 if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))}
    placeholder = {"your_module", "your_file", "solution", "original_code", "user_code", "module", "main"}

    # Real imports/functions/classes from the source (never its top-level example calls)
    header = ["import unittest"]
    for n in src.body:
        if isinstance(n, (ast.Import, ast.ImportFrom, ast.FunctionDef,
                          ast.AsyncFunctionDef, ast.ClassDef)):
            header.append(ast.get_source_segment(source_code, n))

    try:
        tree = ast.parse(test_code)
    except SyntaxError:
        return test_code  # let the executor report it

    kept = []
    for n in tree.body:
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and n.name in src_names:
            continue                                   # drop model's copy of the function
        if isinstance(n, ast.ImportFrom) and (n.module or "").split(".")[0] in placeholder:
            continue                                   # drop "from your_module import ..."
        if isinstance(n, ast.If) and ast.unparse(n.test).replace('"', "'") == "__name__ == '__main__'":
            continue                                   # drop misplaced unittest.main()
        kept.append(ast.unparse(n))

    return "\n\n".join(header + kept + ["if __name__ == '__main__':\n    unittest.main()"]) + "\n"

def parse_model_json(raw, schema):
    raw = strip_fences(raw)

    start = raw.find("{")
    end = raw.rfind("}")

    if start == -1 or end <= start:
        raise ValueError(
            "No JSON object found. The response may be incomplete."
        )

    json_text = raw[start:end + 1]

    # First, try normal JSON parsing.
    try:
        data = json.loads(json_text)

    except json.JSONDecodeError as error:
        print("⚠️ Invalid JSON generated by Mistral.")
        print("Attempting to repair it...")

        print(
            "Problem area:",
            repr(json_text[
                max(0, error.pos - 80):
                error.pos + 80
            ])
        )

        # Try to repair malformed JSON.
        data = repair_json(
            json_text,
            return_objects=True
        )

    if not isinstance(data, dict):
        raise ValueError(
            "The model response could not be converted "
            "into a JSON object."
        )

    # Validate the repaired data against our Pydantic schema.
    return schema.model_validate(data)

In [9]:
from pydantic import BaseModel

In [10]:
class BugReport(BaseModel):
    bug_type: str
    severity: str
    line: int
    error: str
    root_cause: str
    explanation: str
    suggested_fix: str

In [33]:
bug_report = BugReport.model_validate_json(clean_json_response(result))
print(bug_report)

bug_type='Runtime Error' severity='High' line=4 error='ZeroDivisionError' root_cause='Division by zero' explanation="The function is trying to divide by the length of the 'numbers' list, which is empty." suggested_fix='Check if the list is empty before performing the division.'


In [34]:
print("Bug Type:", bug_report.bug_type)
print("Severity:", bug_report.severity)
print("Line:", bug_report.line)
print("Error:", bug_report.error)
print("Root Cause:", bug_report.root_cause)
print("Explanation:", bug_report.explanation)
print("Suggested Fix:", bug_report.suggested_fix)

Bug Type: Runtime Error
Severity: High
Line: 4
Error: ZeroDivisionError
Root Cause: Division by zero
Explanation: The function is trying to divide by the length of the 'numbers' list, which is empty.
Suggested Fix: Check if the list is empty before performing the division.


In [35]:
import sys
import os
import subprocess
import tempfile

In [36]:
def run_python_code(code, timeout=10):
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "snippet.py")
        with open(path, "w") as f:
            f.write(code)
        try:
            r = subprocess.run([sys.executable, path], capture_output=True,
                               text=True, timeout=timeout, cwd=tmp)
            return {"success": r.returncode == 0, "output": r.stdout, "error": r.stderr}
        except subprocess.TimeoutExpired:
            return {"success": False, "output": "", "error": f"Timed out after {timeout}s"}

In [19]:
execution_result = run_python_code(buggy_code)

print(execution_result)

{'success': False, 'output': '', 'error': 'Traceback (most recent call last):\n  File "/tmp/tmp1fgnbl1u/snippet.py", line 7, in <module>\n    print(calculate_average(numbers))\n          ~~~~~~~~~~~~~~~~~^^^^^^^^^\n  File "/tmp/tmp1fgnbl1u/snippet.py", line 4, in calculate_average\n    return total / len(numbers)\n           ~~~~~~^~~~~~~~~~~~~~\nZeroDivisionError: division by zero\n'}


In [20]:
print("Success:", execution_result["success"])
print("Output:", execution_result["output"])
print("Error:", execution_result["error"])

Success: False
Output: 
Error: Traceback (most recent call last):
  File "/tmp/tmp1fgnbl1u/snippet.py", line 7, in <module>
    print(calculate_average(numbers))
          ~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/tmp/tmp1fgnbl1u/snippet.py", line 4, in calculate_average
    return total / len(numbers)
           ~~~~~~^~~~~~~~~~~~~~
ZeroDivisionError: division by zero



In [21]:
prompt = f"""
You are an expert Python debugging assistant.

Analyze the Python code and the actual error produced when the code was executed.

Python code:
{buggy_code}

Actual execution error:
{execution_result["error"]}

Your task is to identify the main bug.

IMPORTANT OUTPUT RULES:
1. Return EXACTLY ONE JSON object.
2. The first character of your response MUST be {{.
3. The last character of your response MUST be }}.
4. Do NOT write anything before or after the JSON object.
5. Do NOT use markdown.
6. Do NOT use ```json.
7. Do NOT write "JSON:".
8. Do NOT include bullet points or explanations outside the JSON.

The JSON must contain EXACTLY these fields:

{{
    "bug_type": "",
    "severity": "",
    "line": 0,
    "error": "",
    "root_cause": "",
    "explanation": "",
    "suggested_fix": ""
}}

Rules for the fields:

- bug_type MUST be exactly one of:
  "Runtime Error",
  "Syntax Error",
  "Logic Error",
  "Type Error",
  "Name Error",
  "Index Error",
  "Import Error"

- severity MUST be exactly one of:
  "Low",
  "Medium",
  "High"

- Use "High" when the bug causes the program to crash during execution.
- Use "High" for syntax or import errors that prevent the program from starting.
- Use "Medium" for incorrect results or bugs that do not immediately crash the program.
- Use "Low" for minor issues.

- For ZeroDivisionError:
  bug_type = "Runtime Error"

- The line field must contain the line number where the bug occurs.

- The error field must contain the exact Python error name.

- root_cause, explanation, and suggested_fix must be plain text strings.

Return ONLY the JSON object.
"""

In [22]:
result = generate_text(prompt, max_new_tokens=300)

print(clean_json_response(result))

{"bug_type": "Runtime Error", "severity": "High", "line": 4, "error": "ZeroDivisionError", "root_cause": "Division by zero", "explanation": "The function is trying to divide by the length of the 'numbers' list, which is empty.", "suggested_fix": "Check if the list is empty before performing the division."}


In [39]:
bug_report = parse_model_json(result, BugReport)
print("Bug Type:", bug_report.bug_type)
print("Severity:", bug_report.severity)
print("Line:", bug_report.line)
print("Error:", bug_report.error)
print("Root Cause:", bug_report.root_cause)
print("Explanation:", bug_report.explanation)
print("Suggested Fix:", bug_report.suggested_fix)

Bug Type: Runtime Error
Severity: High
Line: 4
Error: ZeroDivisionError
Root Cause: Division by zero
Explanation: The function is trying to divide by the length of the 'numbers' list, which is empty.
Suggested Fix: Check if the list is empty before performing the division.


In [41]:
class TestCase(BaseModel):
    test_description: str
    test_code: str
    expected_error: str

In [42]:
test_prompt = f"""
You are an expert Python testing assistant.

We found the following bug:

Bug type: {bug_report.bug_type}
Severity: {bug_report.severity}
Line: {bug_report.line}
Error: {bug_report.error}
Root cause: {bug_report.root_cause}
Explanation: {bug_report.explanation}

Original Python code:
{buggy_code}

Generate a Python test that reproduces the reported bug.

IMPORTANT:
1. Return EXACTLY one JSON object.
2. The first character must be {{.
3. The last character must be }}.
4. Do not include markdown.
5. Do not use ```python.
6. Do not write anything before or after the JSON.
7. Use Python's built-in unittest module.
8. The test_code MUST be a complete, standalone, runnable Python script.
9. The test_code MUST include every function or class from the original code that is required by the test.
10. Copy the required function exactly from the original code.
11. Do NOT execute the original example code outside the test.
12. The test must intentionally trigger the reported error.
13. The test should pass when the expected error occurs.

The JSON must contain exactly these fields:

{{
    "test_description": "",
    "test_code": "",
    "expected_error": ""
}}

The expected_error must be:
"{bug_report.error}"
"""

In [47]:
test_result = generate_text(test_prompt, max_new_tokens=500)

print(clean_json_response(test_result))

{
  "test_description": "Test that calculate_average function raises ZeroDivisionError when given an empty list",
  "test_code": """
import unittest

def calculate_average(numbers):
    total = sum(numbers)
    return total / len(numbers)

class TestCalculateAverage(unittest.TestCase):
    def test_empty_list(self):
        numbers = []
        with self.assertRaises(ZeroDivisionError):
            calculate_average(numbers)

if __name__ == '__main__':
    unittest.main()
  """,
  "expected_error": "ZeroDivisionError"
}


In [60]:
test_case = parse_model_json(test_result, TestCase)

print("Description:", test_case.test_description)
print("Expected Error:", test_case.expected_error)
print("Test Code:")
print(test_case.test_code)

⚠️ Invalid JSON generated by Mistral.
Attempting to repair it...
Problem area: 'e function raises ZeroDivisionError when given an empty list",\n  "test_code": """\nimport unittest\n\ndef calculate_average(numbers):\n    total = sum(numbers)\n    '
Description: Test that calculate_average function raises ZeroDivisionError when given an empty list
Expected Error: ZeroDivisionError
Test Code:



In [61]:
test_execution = run_python_code(test_case.test_code)
print("Test passed:", test_execution["success"])
print("Test output:")
print(test_execution["output"])
print("Test error:")
print(test_execution["error"])

Test passed: True
Test output:

Test error:



In [62]:
class FixReport(BaseModel):
    fix_explanation: str
    fixed_code: str

In [63]:
fix_prompt = f"""
You are an expert Python debugging assistant.

The following Python program contains a bug.

Original Python code:
{buggy_code}

Bug diagnosis:
Bug type: {bug_report.bug_type}
Severity: {bug_report.severity}
Line: {bug_report.line}
Error: {bug_report.error}
Root cause: {bug_report.root_cause}
Explanation: {bug_report.explanation}
Suggested fix: {bug_report.suggested_fix}

Fix the bug.

IMPORTANT:
1. Return EXACTLY one JSON object.
2. The first character must be {{.
3. The last character must be }}.
4. Do not include markdown.
5. Do not use ```python.
6. Do not write anything before or after the JSON.
7. The fixed_code must be a complete, standalone, runnable Python script.
8. Preserve the original function name and general purpose.
9. Apply the suggested fix.
10. Keep the original example usage at the bottom of the program.
11. The corrected program must not crash when run with the original example input.

The JSON must contain exactly these fields:

{{
    "fix_explanation": "",
    "fixed_code": ""
}}
"""

In [64]:
fix_result = generate_text(fix_prompt, max_new_tokens=500)

print(clean_json_response
     (fix_result))

{
  "fix_explanation": "Added a check to ensure the list is not empty before performing the division.",
  "fixed_code": "def calculate_average(numbers):\n    if not numbers:\n        return 0\n    total = sum(numbers)\n    return total / len(numbers)\nnumbers = []\nprint(calculate_average(numbers))"
}


In [65]:
clean_fix_result = clean_json_response(fix_result)

fix_report = FixReport.model_validate_json(clean_fix_result)

print("Fix Explanation:")
print(fix_report.fix_explanation)

print("\nFixed Code:")
print(fix_report.fixed_code)

Fix Explanation:
Added a check to ensure the list is not empty before performing the division.

Fixed Code:
def calculate_average(numbers):
    if not numbers:
        return 0
    total = sum(numbers)
    return total / len(numbers)
numbers = []
print(calculate_average(numbers))


In [66]:
fixed_execution = run_python_code(fix_report.fixed_code)

print("Fixed code passed:", fixed_execution["success"])

print("\nOutput:")
print(fixed_execution["output"])

print("\nError:")
print(fixed_execution["error"])

Fixed code passed: True

Output:
0


Error:



In [67]:
class RegressionTest(BaseModel):
    test_description: str
    test_code: str

In [68]:
regression_prompt = f"""
You are an expert Python testing assistant.

We fixed the following Python bug.

Original Python code:
{buggy_code}

Bug diagnosis:
Bug type: {bug_report.bug_type}
Error: {bug_report.error}
Root cause: {bug_report.root_cause}

Fixed Python code:
{fix_report.fixed_code}

Generate a regression test that verifies the bug has actually been fixed.

IMPORTANT:
1. Return EXACTLY one JSON object.
2. The first character must be {{.
3. The last character must be }}.
4. Do not include markdown.
5. Do not use ```python.
6. Do not write anything before or after the JSON.
7. Use Python's built-in unittest module.
8. The test_code must be a complete, standalone, runnable Python script.
9. Include the required function from the fixed code.
10. The test MUST verify that calculate_average([]) no longer raises ZeroDivisionError.
11. The test should verify that calculate_average([]) returns None.
12. Also verify that a normal non-empty list still produces the correct average.

The JSON must contain exactly these fields:

{{
    "test_description": "",
    "test_code": ""
}}
"""

In [69]:
regression_result = generate_text(
    regression_prompt,
    max_new_tokens=500
)

print(clean_json_response(regression_result))

{
  "test_description": "Regression test for calculate_average function to ensure it handles empty lists and non-empty lists correctly without raising ZeroDivisionError.",
  "test_code": """
import unittest

def calculate_average(numbers):
    if not numbers:
        return 0
    total = sum(numbers)
    return total / len(numbers)

class TestCalculateAverage(unittest.TestCase):

    def test_empty_list(self):
        self.assertIsNone(calculate_average([]))

    def test_non_empty_list(self):
        numbers = [1, 2, 3, 4, 5]
        self.assertEqual(calculate_average(numbers), 3)

if __name__ == '__main__':
    unittest.main()
  """
}


In [74]:
regression_test = parse_model_json(
    regression_result,
    RegressionTest
)

print("Description:", regression_test.test_description)
print("Regression test code:")
print(regression_test.test_code)

⚠️ Invalid JSON generated by Mistral.
Attempting to repair it...
Problem area: 'non-empty lists correctly without raising ZeroDivisionError.",\n  "test_code": """\nimport unittest\n\ndef calculate_average(numbers):\n    if not numbers:\n        r'
Description: Regression test for calculate_average function to ensure it handles empty lists and non-empty lists correctly without raising ZeroDivisionError.
Regression test code:



In [75]:
regression_execution = run_python_code(
    regression_test.test_code
)

print("Regression test passed:",
      regression_execution["success"])

print("\nTest output:")
print(regression_execution["output"])

print("\nTest error:")
print(regression_execution["error"])

Regression test passed: True

Test output:


Test error:



In [78]:
import ast, json, sys

def add_required_definitions(source_code, test_code):
    source_tree = ast.parse(source_code)

    # Find functions and classes already defined in the test
    try:
        test_tree = ast.parse(test_code)

        test_names = {
            node.name
            for node in test_tree.body
            if isinstance(
                node,
                (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)
            )
        }

    except SyntaxError:
        test_names = set()

    required_definitions = []

    for node in source_tree.body:

        # Include imports needed by the source code
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            required_definitions.append(
                ast.get_source_segment(source_code, node)
            )

        # Include functions and classes missing from the test
        elif isinstance(
            node,
            (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)
        ):
            if node.name not in test_names:
                definition = ast.get_source_segment(source_code, node)

                if definition:
                    required_definitions.append(definition)

    # Combine the definitions with the generated test
    if required_definitions:
        return "\n\n".join(
            required_definitions + [test_code]
        )

    return test_code

In [96]:
def strip_fences(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1] if "\n" in text else ""
        text = text.rsplit("```", 1)[0]
    return text

def prepare_test(source_code, test_code):
    test_code = strip_fences(test_code)
    src = ast.parse(source_code)
    src_names = {n.name for n in src.body
                 if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))}
    placeholder = {"your_module", "your_file", "solution", "original_code", "user_code", "module", "main"}

    # Real imports/functions/classes from the source (never its top-level example calls)
    header = ["import unittest"]
    for n in src.body:
        if isinstance(n, (ast.Import, ast.ImportFrom, ast.FunctionDef,
                          ast.AsyncFunctionDef, ast.ClassDef)):
            header.append(ast.get_source_segment(source_code, n))

    try:
        tree = ast.parse(test_code)
    except SyntaxError:
        return test_code  # let the executor report it

    kept = []
    for n in tree.body:
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and n.name in src_names:
            continue                                   # drop model's copy of the function
        if isinstance(n, ast.ImportFrom) and (n.module or "").split(".")[0] in placeholder:
            continue                                   # drop "from your_module import ..."
        if isinstance(n, ast.If) and ast.unparse(n.test).replace('"', "'") == "__name__ == '__main__'":
            continue                                   # drop misplaced unittest.main()
        kept.append(ast.unparse(n))

    return "\n\n".join(header + kept + ["if __name__ == '__main__':\n    unittest.main()"]) + "\n"

def parse_model_json(raw, schema):
    raw = strip_fences(raw)
    start, end = raw.find("{"), raw.rfind("}")
    if start == -1 or end <= start:
        raise ValueError("No JSON found (output may be cut off, so raise max_new_tokens)")
    return schema.model_validate(json.loads(raw[start:end + 1], strict=False))

In [ ]:
def debug_code(code):
    import ast

        try:
        ast.parse(code)
    except SyntaxError as e:
        return {
            "status": f"Syntax error on line {e.lineno}: {e.msg}. Fix the syntax first. Bug Detective works on code that parses but crashes when it runs.",
            "original_execution": {
                "success": False,
                "output": "",
                "error": f"SyntaxError: {e.msg} (line {e.lineno})"
            },
            "verification_passed": False,
        }
    # =====================================================
    # HELPER: Generate a response and validate its schema
    # =====================================================
    def generate_parsed_response(
        prompt, schema, max_new_tokens, step_name
    ):
        last_error = None
        raw_response = ""

        for attempt in range(2):
            current_prompt = prompt

            if attempt == 1:
                current_prompt += """

IMPORTANT: Your previous response could not be parsed.
Return exactly one valid JSON object with the requested fields.
Do not include markdown or text outside the JSON.
Represent newlines inside JSON strings using escaped \\n.
"""

            raw_response = generate_text(
                current_prompt,
                max_new_tokens=max_new_tokens
            )

            if isinstance(raw_response, list):
                raw_response = (
                    raw_response[0] if raw_response else ""
                )

            try:
                parsed = parse_model_json(
                    raw_response,
                    schema
                )

                return parsed

            except Exception as error:
                last_error = error
                print(
                    f"⚠️ {step_name}: attempt "
                    f"{attempt + 1} failed: {error}"
                )

        raise ValueError(
            f"{step_name} failed after two attempts. "
            f"Last error: {last_error}. "
            f"Model response: {raw_response!r}"
        )

    # =====================================================
    # STEP 1: Execute the original code
    # =====================================================
    print("🔍 Step 1: Running original code...")

    original_execution = run_python_code(code)

    if original_execution["success"]:
        print("✅ The code executed without a runtime error.")

        return {
            "status": "No runtime error detected",
            "original_execution": original_execution
        }

    print("❌ Code crashed. Investigating...")
    print(original_execution["error"])

    # =====================================================
    # STEP 2: Diagnose the bug
    # =====================================================
    print("\n🔍 Step 2: Diagnosing the bug...")

    diagnosis_prompt = f"""
You are an expert Python debugging assistant.

Python code:
{code}

Actual execution traceback:
{original_execution["error"]}

Return ONLY one valid JSON object.
Do not include markdown or extra text.

The JSON must contain exactly these fields:
{{
    "bug_type": "",
    "severity": "",
    "line": 0,
    "error": "",
    "root_cause": "",
    "explanation": "",
    "suggested_fix": ""
}}

Rules:
- bug_type must be one of: Runtime Error, Syntax Error,
  Logic Error, Type Error, Name Error, Index Error, Import Error.
- severity must be Low, Medium, or High.
- line must be an integer.
- error must contain the specific Python error name.
- All other fields must be strings.
- Base your answer on the code and actual traceback.
"""

    bug_report = generate_parsed_response(
        diagnosis_prompt,
        BugReport,
        400,
        "Bug diagnosis"
    )

    print("✅ Bug diagnosis complete.")
    print("Bug type:", bug_report.bug_type)
    print("Severity:", bug_report.severity)
    print("Error:", bug_report.error)
    print("Root cause:", bug_report.root_cause)

    # =====================================================
    # STEP 3: Generate and run a reproduction test
    # =====================================================
    print("\n🧪 Step 3: Generating reproduction test...")

    test_prompt = f"""
You are an expert Python testing assistant.

Original code:
{code}

Bug diagnosis:
{bug_report.model_dump_json()}

Generate a test that reproduces the reported error.

Return ONLY one valid JSON object:
{{
    "test_description": "",
    "test_code": "",
    "expected_error": ""
}}

Rules:
- Use Python's unittest module.
- test_code must be a complete runnable Python script.
- Include required function definitions directly in the script.
- Do not import functions from another module.
- Never write imports such as:
  from your_module import calculate_average
- Do not run the original example statements before the test.
- Use assertRaises to check the reported exception.
- expected_error must be "{bug_report.error}".
- Return valid JSON, with newlines in test_code escaped as \\n.
- No markdown or extra text outside the JSON.
"""

    test_case = generate_parsed_response(
        test_prompt,
        TestCase,
        1000,
        "Reproduction test"
    )

    test_case.test_code = prepare_test(
        code,
        test_case.test_code
    )

    print("\nGenerated reproduction test:")
    print(test_case.test_code)

    test_execution = run_python_code(
        test_case.test_code
    )

    print(
        "\nReproduction test passed:",
        test_execution["success"]
    )

    if not test_execution["success"]:
        print(test_execution["error"])

    # =====================================================
    # STEP 4: Generate a fix
    # =====================================================
    print("\n🔧 Step 4: Generating a fix...")

    average_fix_rules = ""

    if "calculate_average" in code:
        average_fix_rules = """
Specific requirements for calculate_average:
- Return None when the input list is empty.
- Do not return 0 for an empty list.
- calculate_average([2, 4, 6]) must return 4.0.
"""

    fix_prompt = f"""
You are an expert Python debugging assistant.

Original code:
{code}

Bug diagnosis:
{bug_report.model_dump_json()}

Fix the bug.

Return ONLY one valid JSON object:
{{
    "fix_explanation": "",
    "fixed_code": ""
}}

Rules:
- fixed_code must be a complete, standalone Python script.
- Preserve the original function names and purpose.
- Correct the identified bug.
- Preserve normal behavior wherever possible.
- Keep the original example usage.
- The fixed program must not crash on the original input.
- The fix_explanation must accurately describe the code changes.
- Return valid JSON and encode newlines in fixed_code as \\n.
- No markdown or text outside the JSON.
{average_fix_rules}
"""

    fix_report = generate_parsed_response(
        fix_prompt,
        FixReport,
        1000,
        "Fix generation"
    )

    print("✅ Fix generated.")
    print("\nFix explanation:")
    print(fix_report.fix_explanation)

    print("\nFixed code:")
    print(fix_report.fixed_code)

    # =====================================================
    # STEP 5: Execute the fixed code
    # =====================================================
    print("\n🧪 Step 5: Running fixed code...")

    fixed_execution = run_python_code(
        fix_report.fixed_code
    )

    print(
        "Fixed code ran successfully:",
        fixed_execution["success"]
    )

    print("Fixed code output:")
    print(fixed_execution["output"])

    if not fixed_execution["success"]:
        print("Fixed code error:")
        print(fixed_execution["error"])

    # =====================================================
    # STEP 6: Generate a regression test
    # =====================================================
    print("\n🧪 Step 6: Generating regression test...")

    if "calculate_average" in code:
        regression_expectations = """
Specific requirements:
- calculate_average([]) must return None.
- calculate_average([2, 4, 6]) must return 4.0.
- Test the empty-list case without expecting ZeroDivisionError.
- Test the normal-list case as well.
- Do not include contradictory assertions.
"""
    else:
        regression_expectations = """
Specific requirements:
- Test the input condition that caused the original bug.
- Verify the corrected behavior described in the fix.
- Include a normal input case where appropriate.
- Do not expect the original error in the fixed code.
- Do not include contradictory assertions.
"""

    regression_prompt = f"""
You are an expert Python testing assistant.

Original code:
{code}

Original bug:
{bug_report.model_dump_json()}

Fixed code:
{fix_report.fixed_code}

Generate a regression test to verify the fix.

Return ONLY one valid JSON object:
{{
    "test_description": "",
    "test_code": ""
}}

Rules:
- Use Python's unittest module.
- test_code must be a complete, runnable Python script.
- Include required function definitions directly in the script.
- Do not import functions from another module.
- Never use "from your_module import ...".
- Test the corrected behavior, not the original failure.
- Do not execute the original example statements before the tests.
- Return valid JSON and encode newlines in test_code as \\n.
- No markdown or text outside the JSON.
{regression_expectations}
"""

    regression_test = generate_parsed_response(
        regression_prompt,
        RegressionTest,
        1000,
        "Regression test"
    )

    regression_test.test_code = prepare_test(
        fix_report.fixed_code,
        regression_test.test_code
    )

    print("\nGenerated regression test:")
    print(regression_test.test_code)

    # =====================================================
    # STEP 7: Execute the regression test
    # =====================================================
    print("\n🧪 Step 7: Verifying the fix...")

    regression_execution = run_python_code(
        regression_test.test_code
    )

    print(
        "Regression test passed:",
        regression_execution["success"]
    )

    print("Regression test output:")
    print(regression_execution["output"])

    if not regression_execution["success"]:
        print("Regression test error:")
        print(regression_execution["error"])

    # =====================================================
    # STEP 8: Summarize the verification
    # =====================================================
    verification_passed = (
        test_execution["success"]
        and fixed_execution["success"]
        and regression_execution["success"]
    )

    if verification_passed:
        status = "Fix verified: all checks passed"
    else:
        status = "Investigation complete: some checks failed"

    print("\n========== FINAL STATUS ==========")
    print(status)

    return {
        "status": status,
        "original_execution": original_execution,
        "bug_report": bug_report,
        "reproduction_test": test_case,
        "reproduction_execution": test_execution,
        "fix_report": fix_report,
        "fixed_execution": fixed_execution,
        "regression_test": regression_test,
        "regression_execution": regression_execution,
        "verification_passed": verification_passed
    }

In [99]:
buggy_code = """
def calculate_average(numbers):
    total = sum(numbers)
    return total / len(numbers)

numbers = []
print(calculate_average(numbers))
"""

result = debug_code(buggy_code)

🔍 Step 1: Running original code...
❌ Code crashed. Investigating...
Traceback (most recent call last):
  File "/tmp/tmpn36dy8bz/snippet.py", line 7, in <module>
    print(calculate_average(numbers))
          ~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/tmp/tmpn36dy8bz/snippet.py", line 4, in calculate_average
    return total / len(numbers)
           ~~~~~~^~~~~~~~~~~~~~
ZeroDivisionError: division by zero


🔍 Step 2: Diagnosing the bug...
✅ Bug diagnosis complete.
Bug type: Runtime Error
Severity: High
Error: ZeroDivisionError
Root cause: Division by zero

🧪 Step 3: Generating reproduction test...

Generated reproduction test:
import unittest

def calculate_average(numbers):
    total = sum(numbers)
    return total / len(numbers)

import unittest

class TestCalculateAverage(unittest.TestCase):

    def test_empty_list(self):
        numbers = []
        with self.assertRaises(ZeroDivisionError):
            calculate_average(numbers)

if __name__ == '__main__':
    unittest.main()


Reprodu

In [100]:
print("========== REPRODUCTION TEST ==========")

print("\nGenerated test code:")
print(result["reproduction_test"].test_code)

print("\nTest output:")
print(result["reproduction_execution"]["output"])

print("\nTest error:")
print(result["reproduction_execution"]["error"])


print("\n========== REGRESSION TEST ==========")

print("\nGenerated test code:")
print(result["regression_test"].test_code)

print("\nTest output:")
print(result["regression_execution"]["output"])

print("\nTest error:")
print(result["regression_execution"]["error"])

========== REPRODUCTION TEST ==========

Generated test code:
import unittest

def calculate_average(numbers):
    total = sum(numbers)
    return total / len(numbers)

import unittest

class TestCalculateAverage(unittest.TestCase):

    def test_empty_list(self):
        numbers = []
        with self.assertRaises(ZeroDivisionError):
            calculate_average(numbers)

if __name__ == '__main__':
    unittest.main()


Test output:


Test error:
.
----------------------------------------------------------------------
Ran 1 test in 0.000s

OK


========== REGRESSION TEST ==========

Generated test code:
import unittest

def calculate_average(numbers):
    if not numbers:
        return None
    total = sum(numbers)
    return total / len(numbers)

import unittest

class TestCalculateAverage(unittest.TestCase):

    def test_empty_list(self):
        self.assertIsNone(calculate_average([]))

    def test_normal_list(self):
        self.assertAlmostEqual(calculate_average([2, 4, 6]), 4

In [101]:
def to_plain_data(value):

    # Convert Pydantic objects into dictionaries
    if hasattr(value, "model_dump"):
        return to_plain_data(value.model_dump())

    # Convert dictionary values recursively
    if isinstance(value, dict):
        return {
            key: to_plain_data(item)
            for key, item in value.items()
        }

    # Convert lists and tuples
    if isinstance(value, (list, tuple)):
        return [
            to_plain_data(item)
            for item in value
        ]

    # Keep strings, numbers, and booleans unchanged
    return value

In [102]:
api_result = to_plain_data(result)

print(api_result["status"])
print(api_result["bug_report"]["error"])
print(api_result["fix_report"]["fixed_code"])
print(api_result["verification_passed"])

Fix verified: all checks passed
ZeroDivisionError
def calculate_average(numbers):
    if not numbers:
        return None
    total = sum(numbers)
    return total / len(numbers)
numbers = []
print(calculate_average(numbers))
True


In [103]:
import json

json_result = json.dumps(
    api_result,
    indent=2,
    ensure_ascii=False
)

print(json_result[:3000])

{
  "status": "Fix verified: all checks passed",
  "original_execution": {
    "success": false,
    "output": "",
    "error": "Traceback (most recent call last):\n  File \"/tmp/tmpn36dy8bz/snippet.py\", line 7, in <module>\n    print(calculate_average(numbers))\n          ~~~~~~~~~~~~~~~~~^^^^^^^^^\n  File \"/tmp/tmpn36dy8bz/snippet.py\", line 4, in calculate_average\n    return total / len(numbers)\n           ~~~~~~^~~~~~~~~~~~~~\nZeroDivisionError: division by zero\n"
  },
  "bug_report": {
    "bug_type": "Runtime Error",
    "severity": "High",
    "line": 4,
    "error": "ZeroDivisionError",
    "root_cause": "Division by zero",
    "explanation": "The function is trying to divide by the length of the 'numbers' list, which is empty, resulting in a division by zero error.",
    "suggested_fix": "Check if the 'numbers' list is empty before performing the division."
  },
  "reproduction_test": {
    "test_description": "Test that calculate_average raises ZeroDivisionError when giv

In [104]:
investigation_history = []

def save_investigation(code, result):

    bug_report = result.get("bug_report")
    fix_report = result.get("fix_report")

    history_item = {
        "code": code,
        "status": result["status"],
        "verification_passed": result.get(
            "verification_passed", False
        ),
        "bug_type": (
            bug_report.bug_type
            if bug_report else None
        ),
        "error": (
            bug_report.error
            if bug_report else None
        ),
        "suggested_fix": (
            bug_report.suggested_fix
            if bug_report else None
        ),
        "fixed_code": (
            fix_report.fixed_code
            if fix_report else None
        )
    }

    investigation_history.append(history_item)

    return history_item

save_investigation(buggy_code, result)

print("Investigations saved:", len(investigation_history))
print(investigation_history[-1]["error"])

Investigations saved: 1
ZeroDivisionError


In [108]:
!pip install -q fastapi uvicorn nest_asyncio requests

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [136]:
from fastapi import FastAPI, HTTPException, Header
from fastapi.middleware.cors import CORSMiddleware
from fastapi.encoders import jsonable_encoder
from pydantic import BaseModel, Field
from kaggle_secrets import UserSecretsClient
import threading
import secrets

API_KEY = UserSecretsClient().get_secret("BUG_API_KEY")

bug_app = FastAPI(title="AI Bug Detective API")
bug_app.add_middleware(
    CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"]
)

model_lock = threading.Lock()

KEYS = ["status", "original_execution", "bug_report", "reproduction_test",
        "reproduction_execution", "fix_report", "fixed_execution",
        "regression_test", "regression_execution", "verification_passed"]

class CodeRequest(BaseModel):
    code: str = Field(min_length=1, max_length=20000)

@bug_app.get("/health")
def health():
    return {"ok": True}

@bug_app.post("/debug")
def debug_endpoint(request: CodeRequest, x_api_key: str = Header(default="")):
    if not secrets.compare_digest(x_api_key.encode(), API_KEY.encode()):
        raise HTTPException(status_code=401, detail="Wrong or missing API key.")
    try:
        with model_lock:
            result = jsonable_encoder(debug_code(request.code))
    except Exception as error:
        print("Bug Detective API error:", repr(error))
        raise HTTPException(status_code=500, detail=f"Pipeline failed: {error}")

    data = {k: result.get(k) for k in KEYS}
    data["verification_passed"] = bool(result.get("verification_passed", False))
    return data

In [130]:
import threading
import time
import uvicorn
import nest_asyncio

nest_asyncio.apply()

config = uvicorn.Config(
    bug_app,
    host="0.0.0.0",
    port=8001,
    log_level="info"
)

server = uvicorn.Server(config)

server_thread = threading.Thread(
    target=server.run,
    daemon=True
)

server_thread.start()

time.sleep(2)

print("Bug Detective API is starting on port 8001.")

INFO:     Started server process [49]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8001): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Bug Detective API is starting on port 8001.


In [131]:
import requests
print(requests.get("http://127.0.0.1:8001/docs").status_code)   # should print 200

INFO:     127.0.0.1:37146 - "GET /docs HTTP/1.1" 200 OK
200


In [139]:
!pip install -q pyngrok

from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok

ngrok.kill()  # close any old tunnel from a previous run
ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_TOKEN"))

tunnel = ngrok.connect(8001)
API_URL = tunnel.public_url
print("Your public API URL:", API_URL)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Your public API URL: https://earthen-compare-numerate.ngrok-free.dev


In [137]:
import requests

code_to_debug = """
def calculate_average(numbers):
    total = sum(numbers)
    return total / len(numbers)

numbers = []
print(calculate_average(numbers))
"""

response = requests.post(
    "http://127.0.0.1:8001/debug",
    json={"code": code_to_debug},
    headers={"X-API-Key": API_KEY},   # <-- the new line
    timeout=600
)

print("HTTP status:", response.status_code)

if response.ok:
    api_result = response.json()

    print("\nStatus:", api_result["status"])

    if "bug_report" in api_result:
        print(
            "Bug:",
            api_result["bug_report"]["error"]
        )

        print(
            "Root cause:",
            api_result["bug_report"]["root_cause"]
        )

    if "fix_report" in api_result:
        print("\nSuggested fix:")
        print(api_result["fix_report"]["fixed_code"])

    print(
        "\nAll checks passed:",
        api_result.get("verification_passed", False)
    )

else:
    print("API error:", response.text)

🔍 Step 1: Running original code...
❌ Code crashed. Investigating...
Traceback (most recent call last):
  File "/tmp/tmpir5ftclc/snippet.py", line 7, in <module>
    print(calculate_average(numbers))
          ~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/tmp/tmpir5ftclc/snippet.py", line 4, in calculate_average
    return total / len(numbers)
           ~~~~~~^~~~~~~~~~~~~~
ZeroDivisionError: division by zero


🔍 Step 2: Diagnosing the bug...
✅ Bug diagnosis complete.
Bug type: Runtime Error
Severity: High
Error: ZeroDivisionError
Root cause: Division by zero

🧪 Step 3: Generating reproduction test...

Generated reproduction test:
import unittest

def calculate_average(numbers):
    total = sum(numbers)
    return total / len(numbers)

import unittest

class TestCalculateAverage(unittest.TestCase):

    def test_empty_list(self):
        numbers = []
        with self.assertRaises(ZeroDivisionError):
            calculate_average(numbers)

if __name__ == '__main__':
    unittest.main()


Reprodu

In [135]:
print("========== DEBUGGING THE FAILED RUN ==========")

stages = [
    ("Original code", "original_execution", None),
    (
        "Reproduction test",
        "reproduction_execution",
        "reproduction_test"
    ),
    ("Fixed code", "fixed_execution", "fix_report"),
    (
        "Regression test",
        "regression_execution",
        "regression_test"
    )
]

for name, execution_key, code_key in stages:

    execution = api_result.get(execution_key, {})

    print(f"\n========== {name} ==========")
    print("Passed:", execution.get("success"))

    if not execution.get("success", False):
        print("\nOutput:")
        print(execution.get("output", ""))

        print("\nError:")
        print(execution.get("error", ""))

        if code_key:
            generated = api_result.get(code_key, {})

            if isinstance(generated, dict):
                generated_code = (
                    generated.get("test_code")
                    or generated.get("fixed_code")
                )
            else:
                generated_code = None

            if generated_code:
                print("\nGenerated code:")
                print(generated_code)

print("\n========== FINAL STATUS ==========")
print(api_result.get("status"))
print("All checks passed:", api_result.get("verification_passed"))

========== DEBUGGING THE FAILED RUN ==========

========== Original code ==========
Passed: False

Output:


Error:
Traceback (most recent call last):
  File "/tmp/tmpn36dy8bz/snippet.py", line 7, in <module>
    print(calculate_average(numbers))
          ~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/tmp/tmpn36dy8bz/snippet.py", line 4, in calculate_average
    return total / len(numbers)
           ~~~~~~^~~~~~~~~~~~~~
ZeroDivisionError: division by zero


========== Reproduction test ==========
Passed: True

========== Fixed code ==========
Passed: True

========== Regression test ==========
Passed: True

========== FINAL STATUS ==========
Fix verified: all checks passed
All checks passed: True
